# Train, cross-validate, engineer, and select

## Objectives

- Fit a preprocessing pipeline and score it with train-fold cross-validation.
- Add two engineered columns and keep a short feature list with `SelectKBest`.
- Leave the validation and test rows closed for later notebooks.

## What we will do

1. Open `lib/pipelines.py`. Each pipeline scales numeric columns and one-hot encodes district, plan, and aisle before classification.
2. Fit a logistic pipeline and print its train-fold ROC-AUC with `cross_val_score`.
3. Write `add_features` for `orders_per_week` and `recency_over_45`, then cross-validate the engineered pipeline.
4. Place `SelectKBest(k=8)` after preprocessing and record which names stay.
5. Optional: cross-validate a plain forest warm-up, then continue to `04_tune.ipynb`.

`cross_val_score` refits on train rows only. The pipeline fits the scaler on the same rows as the classifier, so preprocessing does not use test data.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.model_selection import cross_val_score

from pipelines import make_logistic_pipeline

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
logistic_cv = cross_val_score(
    logistic, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=1
)
print("logistic cv mean", round(float(logistic_cv.mean()), 3))


## Feature engineering

The fit above uses the columns already in the file. Two more columns come from those June fields, one row at a time.

`orders_per_week` is `orders_90d / 13`. `recency_over_45` is 1 when `days_since_last_order` is 45 or more, and 0 otherwise. Do not use `churned`. The same function is in `lib/features.py`. Notebooks 04 and 05 use that copy so the saved model still has the columns after a kernel restart.

Write `add_features` so it returns a copy with those two columns added. The next cell puts that function in front of the scaler, inside a pipeline. `StratifiedKFold` keeps a similar churn rate in each train fold. The shuffle is fixed with seed 42.

On the train rows, expect the mean of `orders_per_week` near 0.41 and the share of `recency_over_45` near 0.51. The cross-validation ROC-AUC should stay near 0.78, close to the logistic mean above. This is a train-fold score. Notebook 04 compares models on the validation rows. Do not save this pipeline.


In [ ]:
def add_features(frame):
    raise NotImplementedError("Add orders_per_week and recency_over_45.")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from schema import CATEGORICAL_FEATURES, NUMERIC_FEATURES

engineered = add_features(X_train)
needed = {"orders_per_week", "recency_over_45"}
if not needed.issubset(engineered.columns):
    raise NotImplementedError("Add orders_per_week and recency_over_45.")
print("train orders_per_week mean", round(float(engineered["orders_per_week"].mean()), 3))
print("train recency_over_45 share", round(float(engineered["recency_over_45"].mean()), 3))

numeric = list(NUMERIC_FEATURES) + ["orders_per_week", "recency_over_45"]
fe_logistic = Pipeline(
    steps=[
        ("add", FunctionTransformer(add_features, validate=False)),
        (
            "prep",
            ColumnTransformer(
                transformers=[
                    ("num", StandardScaler(), numeric),
                    (
                        "cat",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        list(CATEGORICAL_FEATURES),
                    ),
                ]
            ),
        ),
        ("model", LogisticRegression(max_iter=500, solver="lbfgs")),
    ]
)
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fe_cv = cross_val_score(fe_logistic, X_train, y_train, cv=folds, scoring="roc_auc", n_jobs=1)
print("engineered cv mean", round(float(fe_cv.mean()), 3))


## Feature selection

`SelectKBest(f_classif, k=8)` sits after `add_features`, the scaler, and the one-hot encoder. It fits on the train rows only. `RFE` is the other common tool. This table is small, so `SelectKBest` is enough.

Eight names should come back. `orders_per_week` should be one of them. The train-fold ROC-AUC should be near 0.79. Notebook 04 fits this same pipeline for logistic regression, a random forest, and an SVM, then picks one model on the validation rows.

Do not overwrite the model you will choose in notebook 04 with this pipeline.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from pipelines import make_engineered_preprocessor

selector = Pipeline(
    steps=[
        ("add", FunctionTransformer(add_features, validate=False)),
        ("prep", make_engineered_preprocessor()),
        ("select", None),
        ("model", LogisticRegression(max_iter=500, solver="lbfgs")),
    ]
)
selected_count = None
orders_kept = None


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

if selector.named_steps["select"] is None or selected_count is None or orders_kept is None:
    raise NotImplementedError("Set SelectKBest, then record the count and whether orders_per_week stayed.")
selector.fit(X_train, y_train)
names = selector.named_steps["prep"].get_feature_names_out()
mask = selector.named_steps["select"].get_support()
picked = list(names[mask])
select_cv = cross_val_score(
    selector,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=1,
)
print(picked)
print("selected", int(selected_count), "orders_per_week kept", bool(orders_kept))
print("select cv mean", round(float(select_cv.mean()), 3))


### Exercises

Fit `make_forest_pipeline` the same way. That pipeline is a plain forest warm-up. It does not add the engineered columns or `SelectKBest`. It is not the forest candidate in notebook 04. Use `cross_val_score` on `X_train` and `y_train` only, with `cv=3` and `scoring="roc_auc"`. Store the mean in `plain_forest_cv_mean`.

Leave the validation rows for notebook 04.


In [ ]:
from sklearn.model_selection import cross_val_score

from pipelines import make_forest_pipeline

forest = None
plain_forest_cv_mean = None


In [ ]:
if plain_forest_cv_mean is None:
    raise NotImplementedError("Finish the forest cross-validation exercise.")
print("forest cv mean", round(float(plain_forest_cv_mean), 3))
